In [7]:
import random
from collections import defaultdict

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

GRID = 6                 # red de calles de GRID x GRID intersecciones
N_VEHICLES = 10
MAX_TICKS = 80            # tope de seguridad para detener la simulación
random.seed(7)

NODES = [(r, c) for r in range(GRID) for c in range(GRID)]


def neighbors(node):
    """Intersecciones conectadas ortogonalmente a `node`."""
    r, c = node
    cand = [(r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)]
    return [n for n in cand if 0 <= n[0] < GRID and 0 <= n[1] < GRID]


def dist(a, b):
    """Distancia Manhattan sobre la red de calles."""
    return abs(a[0] - b[0]) + abs(a[1] - b[1])


In [8]:
class Vehicle:
    def __init__(self, vid, origin, destination):
        self.vid = vid
        self.origin = origin
        self.destination = destination
        self.pos = origin
        self.route = [origin]
        self.time = 0          # turnos transcurridos hasta llegar
        self.waits = 0          # veces que esperó por congestión
        self.arrived = (origin == destination)

    def next_move(self):
        """Elige la intersección vecina que más reduce la distancia al destino."""
        options = neighbors(self.pos)
        best = min(dist(n, self.destination) for n in options)
        best_options = [n for n in options if dist(n, self.destination) == best]
        return random.choice(best_options)


class TrafficWorld:
    def __init__(self, n_vehicles=N_VEHICLES):
        self.conflicts = 0
        self.tick = 0
        self.vehicles = []
        for i in range(n_vehicles):
            origin = random.choice(NODES)
            destination = random.choice(NODES)
            while destination == origin:
                destination = random.choice(NODES)
            self.vehicles.append(Vehicle(i, origin, destination))

    def active_vehicles(self):
        return [v for v in self.vehicles if not v.arrived]

    def step(self):
        """Avanza un turno: cada vehículo activo propone un movimiento y se
        resuelven las congestiones antes de aplicar los cambios de posición."""
        active = self.active_vehicles()
        if not active:
            return False

        proposals = {v: v.next_move() for v in active}

        target_groups = defaultdict(list)
        for v, target in proposals.items():
            target_groups[target].append(v)

        winners = {}
        for target, group in target_groups.items():
            winner = group[0] if len(group) == 1 else random.choice(group)
            winners[winner] = target
            for v in group:
                if v is not winner:
                    v.waits += 1
                    self.conflicts += 1

        for v in active:
            v.time += 1
            if v in winners:
                v.pos = winners[v]
                v.route.append(v.pos)
                if v.pos == v.destination:
                    v.arrived = True
            else:
                v.route.append(v.pos)   # se queda esperando por congestión

        self.tick += 1
        return True


In [9]:
world = TrafficWorld()

# snapshot inicial (tick 0) para la animación
history = [{v.vid: (v.pos, False) for v in world.vehicles}]

while world.tick < MAX_TICKS and world.active_vehicles():
    active_before = {v.vid for v in world.active_vehicles()}
    world.step()
    snapshot = {}
    for v in world.vehicles:
        waited = (v.vid in active_before and len(v.route) >= 2
                  and v.route[-1] == v.route[-2])
        snapshot[v.vid] = (v.pos, waited)
    history.append(snapshot)

print(f"Simulación terminada en {world.tick} turnos "
      f"({len(history)} snapshots registrados).")


Simulación terminada en 9 turnos (10 snapshots registrados).


In [10]:
PAL = plt.cm.tab10.colors + plt.cm.Set3.colors
colors = {v.vid: PAL[v.vid % len(PAL)] for v in world.vehicles}

fig, ax = plt.subplots(figsize=(6.5, 6.5))
fig.patch.set_facecolor("#1a202c")


def draw_network(ax):
    ax.set_facecolor("#2d3748")
    for r in range(GRID):
        for c in range(GRID):
            for nr, nc in neighbors((r, c)):
                if (nr, nc) > (r, c):
                    ax.plot([c, nc], [r, nr], color="#4a5568", lw=1.2, zorder=1)
    xs = [c for r, c in NODES]
    ys = [r for r, c in NODES]
    ax.scatter(xs, ys, s=18, color="#718096", zorder=2)


def init():
    ax.clear()
    ax.set_xlim(-0.5, GRID - 0.5)
    ax.set_ylim(-0.5, GRID - 0.5)
    ax.set_aspect("equal")
    ax.set_xticks([]); ax.set_yticks([])
    draw_network(ax)
    return []


def update(frame):
    ax.clear()
    ax.set_xlim(-0.5, GRID - 0.5)
    ax.set_ylim(-0.5, GRID - 0.5)
    ax.set_aspect("equal")
    ax.set_xticks([]); ax.set_yticks([])
    draw_network(ax)

    snapshot = history[frame]
    waiting_now = sum(1 for _, w in snapshot.values() if w)
    ax.set_title(f"Turno {frame:03d}  |  Esperando por congestión: {waiting_now}",
                 fontsize=10, color="#e2e8f0", pad=8)

    for v in world.vehicles:
        r, c = v.origin
        ax.scatter([c], [r], marker="^", s=60, color=colors[v.vid], alpha=0.35, zorder=3)
        r, c = v.destination
        ax.scatter([c], [r], marker="*", s=110, color=colors[v.vid], alpha=0.6, zorder=3)

        pos, waiting = snapshot[v.vid]
        r, c = pos
        arrived_by_now = pos == v.destination
        face = colors[v.vid]
        edge = "#ff3030" if waiting else "#1a1a1a"
        alpha = 0.35 if arrived_by_now else 1.0
        ax.scatter([c], [r], s=140, color=face, edgecolor=edge, linewidth=1.6,
                   alpha=alpha, zorder=4)
        ax.annotate(str(v.vid), (c, r), color="white", fontsize=7,
                    ha="center", va="center", zorder=5)
    return []


anim = FuncAnimation(fig, update, frames=len(history), init_func=init,
                      blit=False, interval=250)
plt.close()
HTML(anim.to_jshtml())


In [11]:
header = f"{'Veh.':>4} | {'Origen':>8} | {'Destino':>8} | {'Tiempo':>6} | {'Esperas':>7} | {'Llegó':>5} | Ruta"
print(header)
print("-" * len(header))
for v in world.vehicles:
    tiempo = str(v.time) if v.arrived else "—"
    llego = "Sí" if v.arrived else "No"
    ruta = " → ".join(str(p) for p in v.route)
    print(f"{v.vid:>4} | {str(v.origin):>8} | {str(v.destination):>8} | "
          f"{tiempo:>6} | {v.waits:>7} | {llego:>5} | {ruta}")

n_arrived = sum(v.arrived for v in world.vehicles)
success_rate = 100 * n_arrived / len(world.vehicles)

print(f"\nConflictos/congestiones totales durante la simulación: {world.conflicts}")
print(f"Vehículos que llegaron a destino: {n_arrived}/{len(world.vehicles)} "
      f"({success_rate:.1f}%)")


Veh. |   Origen |  Destino | Tiempo | Esperas | Llegó | Ruta
------------------------------------------------------------
   0 |   (3, 2) |   (1, 3) |      3 |       0 |    Sí | (3, 2) → (2, 2) → (2, 3) → (1, 3)
   1 |   (4, 1) |   (0, 3) |      7 |       1 |    Sí | (4, 1) → (3, 1) → (2, 1) → (2, 2) → (2, 2) → (1, 2) → (0, 2) → (0, 3)
   2 |   (0, 4) |   (5, 4) |      5 |       0 |    Sí | (0, 4) → (1, 4) → (2, 4) → (3, 4) → (4, 4) → (5, 4)
   3 |   (1, 0) |   (3, 5) |      9 |       2 |    Sí | (1, 0) → (2, 0) → (2, 0) → (2, 1) → (2, 1) → (3, 1) → (3, 2) → (3, 3) → (3, 4) → (3, 5)
   4 |   (0, 3) |   (5, 2) |      7 |       1 |    Sí | (0, 3) → (0, 2) → (1, 2) → (1, 2) → (2, 2) → (3, 2) → (4, 2) → (5, 2)
   5 |   (2, 1) |   (0, 2) |      3 |       0 |    Sí | (2, 1) → (1, 1) → (0, 1) → (0, 2)
   6 |   (0, 5) |   (4, 3) |      7 |       1 |    Sí | (0, 5) → (1, 5) → (2, 5) → (2, 4) → (2, 4) → (2, 3) → (3, 3) → (4, 3)
   7 |   (4, 2) |   (0, 4) |      7 |       1 |    Sí | (4, 2) → (3,